# 03, Intent and diagnosis LLM (the Ollama fine-tune)

**NOT EXECUTED IN THIS ENVIRONMENT.** This sandbox has no running Ollama server, no GPU, and no
Hugging Face auth for the gated `meta-llama/Meta-Llama-3.1-8B-Instruct` weights that Track B
needs. The code below is complete and intended to run as-is on a machine that has those three
things, it is not a stub or placeholder, just unexecuted. See `05-llm-finetuning-ollama.md` for
the full writeup of what running it should produce and how to roll back if it doesn't.

**Inputs:** `data/processed/{train,val,test}.csv` (notebook 01), `models/prompts/llm/diagnosis.txt`

**Outputs:** Track A baseline metrics; Track B a merged, quantised, `ollama create`-registered
model (`cst-diagnosis:v1`) plus `evaluation/reports/llm_comparison.md`.

**Two tracks, run both and compare:**
- **Track A** (no training): prompt engineering + few-shot for `llama3.1:8b-instruct` through
  Ollama, `format: json`. This is the baseline v1 ships with if training fails or is skipped.
- **Track B** (LoRA): QLoRA fine-tune, merge, convert to GGUF, register with Ollama.

In [ ]:
import os, sys
from pathlib import Path

# Notebook hygiene: always run from the v1 root, never hard-code absolute d:\... paths.
_here = Path.cwd()
_v1_root = _here if (_here / "pyproject.toml").exists() else _here.parents[0]
os.chdir(_v1_root)
sys.path.insert(0, str(_v1_root))

# Notebooks that touch libs.platform.registry/config must pin a profile explicitly, the
# process env has no APP_PROFILE by default, which silently falls back to "cpu" (real Ollama
# calls, no stub) and produces confusing failures/latency. Pin stub here; override the env var
# before importing libs.platform.config in the rare notebook that needs otherwise.
os.environ.setdefault("APP_PROFILE", "stub")
print("cwd:", Path.cwd(), "| APP_PROFILE:", os.environ["APP_PROFILE"])

In [ ]:
import json

import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
# `intent` is already a column from notebook 01 (Bitext's 27 intents + the telecom supplement's).
train_df[["text", "department", "intent"]].head()

## Building the intent/diagnosis instruction dataset

Bitext's `intent` column plus the telecom supplement's give the `intent` target; `fault` is
only meaningful for the telecom rows (Bitext has no fault concept), Bitext rows get
`fault: null` and `department` doubles as the coarse target for those.

In [ ]:
_TELECOM_INTENT_TO_FAULT = {
    "no_internet_connection": "fault_power_supply",
    "router_led_issue": "fault_power_supply",
    "slow_internet_speed": "fault_intermittent_connection",
    "intermittent_connection": "fault_intermittent_connection",
    "no_dial_tone": "fault_no_dial_tone",
    "line_synchronization_issue": "fault_line_sync",
    "area_outage_report": "fault_line_sync",
    "schedule_technician_visit": None,
    "missed_technician_appointment": None,
    "equipment_installation_request": None,
    "cabling_issue_onsite": "fault_cabling",
    "technician_visit_status": None,
}

SYSTEM_PROMPT = (
    "You are a telecom customer-support intent and fault classifier. Respond as JSON: "
    '{"intent": str, "fault": str|null, "department": str}'
)


def to_chat_example(row) -> dict:
    fault = _TELECOM_INTENT_TO_FAULT.get(row["intent"])
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["text"]},
            {"role": "assistant", "content": json.dumps({"intent": row["intent"], "fault": fault, "department": row["department"]})},
        ]
    }


train_examples = [to_chat_example(row) for _, row in train_df.iterrows()]
val_examples = [to_chat_example(row) for _, row in val_df.iterrows()]

Path("data/processed").mkdir(parents=True, exist_ok=True)
with open("data/processed/llm_train.jsonl", "w", encoding="utf-8") as f:
    for ex in train_examples:
        f.write(json.dumps(ex) + "\n")
with open("data/processed/llm_val.jsonl", "w", encoding="utf-8") as f:
    for ex in val_examples:
        f.write(json.dumps(ex) + "\n")

len(train_examples), len(val_examples)

## Track A: prompt engineering + few-shot, no training

In [ ]:
import httpx

OLLAMA_BASE_URL = "http://localhost:11434"
BASE_MODEL = "llama3.1:8b-instruct-q4_K_M"

FEW_SHOT_EXAMPLES = train_df.sample(6, random_state=22)[["text", "intent", "department"]].to_dict("records")


def build_few_shot_prompt(text: str) -> str:
    shots = "\n\n".join(
        f'Ticket: {ex["text"]}\nJSON: {{"intent": "{ex["intent"]}", "department": "{ex["department"]}"}}'
        for ex in FEW_SHOT_EXAMPLES
    )
    return f"{shots}\n\nTicket: {text}\nJSON:"


def classify_via_ollama(text: str, model: str = BASE_MODEL) -> dict:
    resp = httpx.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={
            "model": model,
            "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": build_few_shot_prompt(text)}],
            "format": "json",
            "stream": False,
            "options": {"temperature": 0.1},
        },
        timeout=120.0,
    )
    resp.raise_for_status()
    return json.loads(resp.json()["message"]["content"])


# Evaluate Track A on a held-out sample of the val split.
eval_sample = val_df.sample(min(200, len(val_df)), random_state=22)
track_a_results = []
for _, row in eval_sample.iterrows():
    try:
        parsed = classify_via_ollama(row["text"])
        track_a_results.append({
            "intent_correct": parsed.get("intent") == row["intent"],
            "department_correct": parsed.get("department") == row["department"],
            "json_valid": True,
        })
    except Exception:
        track_a_results.append({"intent_correct": False, "department_correct": False, "json_valid": False})

track_a_df = pd.DataFrame(track_a_results)
track_a_summary = {
    "intent_accuracy": track_a_df["intent_correct"].mean(),
    "department_accuracy": track_a_df["department_correct"].mean(),
    "json_validity_rate": track_a_df["json_valid"].mean(),
}
track_a_summary

## Track B: QLoRA fine-tune

4-bit, r=16, alpha=32, lr=2e-4, 2 epochs, batch size 1 x grad-accum 16, max_seq_len=1024.

**Pitfalls (see `docs/00-IMPLEMENTATION-PLAN.md` §9):** needs ~10-12 GB VRAM for the 8B base, if
the GPU is smaller, switch `BASE_MODEL_ID` to `Qwen/Qwen2.5-3B-Instruct` or
`meta-llama/Llama-3.2-3B-Instruct`; the Llama 3.1 weights are gated on Hugging Face (`huggingface-cli
login` and accept the licence first); GGUF conversion needs the **merged** model, not the bare
adapter; `bitsandbytes` on Windows needs a recent wheel, if it won't install, run this notebook
in WSL2 or Colab and copy the resulting GGUF back.

In [ ]:
BASE_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"  # or a 3B fallback, see pitfalls above
ADAPTER_DIR = Path("models/artifacts/cst-diagnosis-lora")
MERGED_DIR = Path("models/artifacts/cst-diagnosis-merged")
GGUF_PATH = Path("models/artifacts/cst-diagnosis-q4_K_M.gguf")

import torch
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer  # part of the `datasets`/`transformers` ecosystem; add to [train] if missing

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

train_ds = load_dataset("json", data_files="data/processed/llm_train.jsonl", split="train")
val_ds = load_dataset("json", data_files="data/processed/llm_val.jsonl", split="train")


def format_example(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}


train_ds = train_ds.map(format_example)
val_ds = val_ds.map(format_example)

training_args = TrainingArguments(
    output_dir=str(ADAPTER_DIR),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,  # if OOM on a smaller GPU, per the plan's pitfalls
)

trainer = SFTTrainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    dataset_text_field="text", max_seq_length=1024,
)
trainer.train()
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

## Merge, convert to GGUF, quantise, register with Ollama

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR).merge_and_unload()  # merged, not the bare adapter
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

In [ ]:
# Requires a local llama.cpp checkout with convert_hf_to_gguf.py and the `llama-quantize` binary.
import subprocess

subprocess.run(
    ["python", "llama.cpp/convert_hf_to_gguf.py", str(MERGED_DIR), "--outfile", "models/artifacts/cst-diagnosis-f16.gguf"],
    check=True,
)
subprocess.run(
    ["llama.cpp/llama-quantize", "models/artifacts/cst-diagnosis-f16.gguf", str(GGUF_PATH), "q4_K_M"], check=True
)

In [ ]:
modelfile = f'''FROM {GGUF_PATH}
TEMPLATE """{{{{ if .System }}}}<|start_header_id|>system<|end_header_id|>\n\n{{{{ .System }}}}<|eot_id|>{{{{ end }}}}<|start_header_id|>user<|end_header_id|>\n\n{{{{ .Prompt }}}}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"""
PARAMETER temperature 0.2
SYSTEM """{SYSTEM_PROMPT}"""
'''
Path("models/Modelfile").write_text(modelfile, encoding="utf-8")

subprocess.run(["ollama", "create", "cst-diagnosis:v1", "-f", "models/Modelfile"], check=True)
print("registered cst-diagnosis:v1, set llm_model=cst-diagnosis:v1 in .env to use it")

## Evaluation: base vs fine-tuned

In [ ]:
track_b_results = []
for _, row in eval_sample.iterrows():
    try:
        parsed = classify_via_ollama(row["text"], model="cst-diagnosis:v1")
        track_b_results.append({
            "intent_correct": parsed.get("intent") == row["intent"],
            "department_correct": parsed.get("department") == row["department"],
            "json_valid": True,
        })
    except Exception:
        track_b_results.append({"intent_correct": False, "department_correct": False, "json_valid": False})

track_b_df = pd.DataFrame(track_b_results)
track_b_summary = {
    "intent_accuracy": track_b_df["intent_correct"].mean(),
    "department_accuracy": track_b_df["department_correct"].mean(),
    "json_validity_rate": track_b_df["json_valid"].mean(),
}

comparison_md = f'''# LLM comparison: llama3.1:8b (Track A) vs cst-diagnosis:v1 (Track B)

| metric | llama3.1:8b (prompted) | cst-diagnosis:v1 (LoRA) |
|---|---|---|
| intent accuracy | {track_a_summary["intent_accuracy"]:.3f} | {track_b_summary["intent_accuracy"]:.3f} |
| department accuracy | {track_a_summary["department_accuracy"]:.3f} | {track_b_summary["department_accuracy"]:.3f} |
| JSON validity rate | {track_a_summary["json_validity_rate"]:.3f} | {track_b_summary["json_validity_rate"]:.3f} |

Rollback: set `llm_model=llama3.1:8b-instruct-q4_K_M` in `.env` (Track A) if Track B underperforms
or `cst-diagnosis:v1` is not registered.
'''
Path("evaluation/reports").mkdir(parents=True, exist_ok=True)
Path("evaluation/reports/llm_comparison.md").write_text(comparison_md, encoding="utf-8")
print(comparison_md)